In [ ]:
import json
from collections import defaultdict
from typing import List, Dict, Tuple

In [ ]:
ALIGNED_FILE = "aligned_en_vi.json"
EN_SENSE_FILE = "en_sense_list.json"
VI_SENSE_FILE = "vi_sense_list.json"

with open(ALIGNED_FILE, "r", encoding="utf-8") as f:
    aligned_data = json.load(f)

with open(EN_SENSE_FILE, "r", encoding="utf-8") as f:
    en_sense_list = json.load(f)

with open(VI_SENSE_FILE, "r", encoding="utf-8") as f:
    vi_sense_list = json.load(f)

print("Aligned sentences:", len(aligned_data))

Aligned sentences: 49275


In [ ]:
en_lemma_to_senses = defaultdict(list)

for lemma, pos_dict in en_sense_list.items():
    for pos, senses in pos_dict.items():
        for s in senses:
            en_lemma_to_senses[lemma.lower()].append(s["id"])

In [ ]:
en_to_vi_sense = {}

for vi_lemma, pos_dict in vi_sense_list.items():
    for pos, senses in pos_dict.items():
        for s in senses:
            en_id = s.get("matching_en_sense")
            if en_id:
                en_to_vi_sense[en_id] = s["id"]

print("EN→VI sense mappings:", len(en_to_vi_sense))

EN→VI sense mappings: 5053


In [ ]:
def assign_senses_rule_based(
    en_tokens: List[str],
    vi_tokens: List[str],
    alignments: List[List[int]]
) -> Tuple[List[str], List[str], List[Dict]]:
    """
    Returns:
      en_labels, vi_labels, unresolved_cases
    """

    en_labels = ["O"] * len(en_tokens)
    vi_labels = ["O"] * len(vi_tokens)
    unresolved = []

    for en_idx, vi_idx in alignments:
        if en_idx >= len(en_tokens) or vi_idx >= len(vi_tokens):
            continue

        en_tok = en_tokens[en_idx].lower()
        vi_tok = vi_tokens[vi_idx].lower()

        # EN token không có sense
        if en_tok not in en_lemma_to_senses:
            unresolved.append({
                "reason": "en_token_not_in_sense_list",
                "en_token": en_tok,
                "vi_token": vi_tok,
                "en_index": en_idx,
                "vi_index": vi_idx
            })
            continue

        candidate_en_senses = en_lemma_to_senses[en_tok]

        matched = False
        for en_sense_id in candidate_en_senses:
            if en_sense_id in en_to_vi_sense:
                vi_sense_id = en_to_vi_sense[en_sense_id]
                en_labels[en_idx] = en_sense_id
                vi_labels[vi_idx] = vi_sense_id
                matched = True
                break

        if not matched:
            unresolved.append({
                "reason": "no_matching_vi_sense",
                "en_token": en_tok,
                "vi_token": vi_tok,
                "candidate_en_senses": candidate_en_senses,
                "en_index": en_idx,
                "vi_index": vi_idx
            })

    return en_labels, vi_labels, unresolved

In [ ]:
valid_samples = []
unfinished_samples = []

for sent in aligned_data:
    en_tokens = sent["en"]["tokens"]
    vi_tokens = sent["vi"]["tokens"]
    alignments = sent["alignment"]

    en_labels, vi_labels, unresolved = assign_senses_rule_based(
        en_tokens, vi_tokens, alignments
    )

    if any(l != "O" for l in en_labels):
        valid_samples.append({
            "en": {
                "tokens": en_tokens,
                "labels": en_labels
            },
            "vi": {
                "tokens": vi_tokens,
                "labels": vi_labels
            }
        })
    else:
        unfinished_samples.append({
            "en_tokens": en_tokens,
            "vi_tokens": vi_tokens,
            "alignment": alignments,
            "unresolved": unresolved
        })

print("Valid samples:", len(valid_samples))
print("Unfinished samples:", len(unfinished_samples))

Valid samples: 40412
Unfinished samples: 8863


In [ ]:
with open("parallel_sense_ok.jsonl", "w", encoding="utf-8") as f:
    for s in valid_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

with open("parallel_sense_unfinished.jsonl", "w", encoding="utf-8") as f:
    for s in unfinished_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print("Saved:")
print(" - parallel_sense_ok.jsonl")
print(" - parallel_sense_unfinished.jsonl")

Saved:
 - parallel_sense_ok.jsonl
 - parallel_sense_unfinished.jsonl


In [ ]:
import json

INPUT_FILE = "parallel_sense_unfinished.jsonl"
OUTPUT_FILE = "parallel_sense_unfinished_clean.jsonl"

cleaned_samples = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        sample = json.loads(line)

        en_tokens = sample["en_tokens"]
        vi_tokens = sample["vi_tokens"]
        alignment = sample["alignment"]
        unresolved = sample.get("unresolved", [])

        # labels mặc định cho team gán
        en_labels = ["O"] * len(en_tokens)
        vi_labels = ["O"] * len(vi_tokens)

        candidates = []

        for u in unresolved:
            if "candidate_en_senses" not in u:
                continue

            candidates.append({
                "en_index": u["en_index"],
                "vi_index": u["vi_index"],
                "en_token": u["en_token"],
                "vi_token": u["vi_token"],
                "candidate_en_senses": u["candidate_en_senses"]
            })

        # chỉ giữ sample có candidate thực sự
        if candidates:
            cleaned_samples.append({
                "en": {
                    "tokens": en_tokens,
                    "labels": en_labels
                },
                "vi": {
                    "tokens": vi_tokens,
                    "labels": vi_labels
                },
                "alignment": alignment,
                "candidates": candidates
            })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for s in cleaned_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print("Saved cleaned unfinished dataset:")
print(f" - {OUTPUT_FILE}")
print("Total samples:", len(cleaned_samples))

Saved cleaned unfinished dataset:
 - parallel_sense_unfinished_clean.jsonl
Total samples: 2797


In [ ]:
import json

INPUT_JSONL = "parallel_sense_unfinished_clean.jsonl"
OUTPUT_JSON = "parallel_sense_unfinished_clean.json"

data = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Converted:")
print(f" - {INPUT_JSONL}  →  {OUTPUT_JSON}")
print("Total samples:", len(data))

Converted:
 - parallel_sense_unfinished_clean.jsonl  →  parallel_sense_unfinished_clean.json
Total samples: 2797


In [ ]:
import json

INPUT_FILE = "input.json"
OUTPUT_FILE = "filtered.json"

def has_non_O_label(labels):
    return any(label != "O" for label in labels)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

filtered_data = []

for obj in data:
    en_labels = obj.get("en", {}).get("labels", [])
    vi_labels = obj.get("vi", {}).get("labels", [])

    if has_non_O_label(en_labels) or has_non_O_label(vi_labels):
        filtered_data.append(obj)

print(f"Original size : {len(data)}")
print(f"Filtered size : {len(filtered_data)}")
print(f"Removed       : {len(data) - len(filtered_data)}")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

Original size : 878
Filtered size : 845
Removed       : 33


In [ ]:
import json

INPUT_FILE = "filtered.json"
OUTPUT_FILE = "filtered.jsonl"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for obj in data:
        line = {
            "en": {
                "tokens": obj["en"]["tokens"],
                "labels": obj["en"]["labels"]
            },
            "vi": {
                "tokens": obj["vi"]["tokens"],
                "labels": obj["vi"]["labels"]
            }
        }
        f.write(json.dumps(line, ensure_ascii=False) + "\n")

print(f"Converted {len(data)} samples to JSONL")

Converted 845 samples to JSONL
